In [ ]:
# ==============================================================================
# NOTEBOOK: PROVA DE CONCEITO - AGENTE DE DESCOBERTA DE FERRAMENTAS
# FASE 1
#
# OBJETIVO: Demonstrar o fluxo completo de como um LLM pode usar um
#           catálogo de ferramentas para escolher a ação correta a partir de
#           uma pergunta do usuário.
#
# TECNOLOGIAS SIMULADAS:
# - DynamoDB/Postgres (Catálogo): Simulado com um dicionário Python.
# - pgvector/FAISS (Banco Vetorial): Simulado com a biblioteca FAISS-CPU em memória.
# - Amazon Bedrock (Embeddings): Requer credenciais AWS configuradas.
# - LLM (Raciocínio): Requer credenciais AWS para Anthropic Claude.
# ==============================================================================


In [5]:
!pip install boto3 langchain-aws faiss-cpu numpy langchain

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 KB 2.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 KB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 KB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 KB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 KB 27.8 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import boto3
from langchain_aws import BedrockEmbeddings, ChatBedrock

In [3]:
# ==============================================================================
# ETAPA 1-3: MODELAGEM, CATÁLOGO E CADASTRO DE FERRAMENTAS
# Simulação de um banco de dados (DynamoDB/Postgres) com nossas ferramentas.
# O mais importante aqui é a qualidade do campo "description_for_llm".
# ==============================================================================

print("--- 1. Definindo o Catálogo de Ferramentas (Simulação do DynamoDB) ---")

tool_catalog = {
    "tool_001": {
        "tool_name": "Calcular Net Promoter Score (NPS)",
        "description_for_llm": "Use esta ferramenta para obter a pontuação de Net Promoter Score (NPS) de um produto específico. A ferramenta pode receber um período de tempo (datas de início e fim) para filtrar a análise. É ideal para medir a lealdade e satisfação do cliente.",
        "api_spec": {
            "name": "calculate_nps",
            "description": "Calcula o NPS de um produto.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string", "description": "O nome do produto para análise."},
                    "start_date": {"type": "string", "description": "Data de início no formato YYYY-MM-DD."},
                    "end_date": {"type": "string", "description": "Data de fim no formato YYYY-MM-DD."},
                },
                "required": ["product_name"]
            }
        }
    },
    "tool_002": {
        "tool_name": "Buscar Informações de Usuário",
        "description_for_llm": "Use esta ferramenta para recuperar informações detalhadas de um usuário, como nome, data de cadastro e status da conta, usando o email como identificador. É a ferramenta certa para quando a pergunta envolve um usuário específico.",
        "api_spec": {
            "name": "get_user_by_email",
            "description": "Busca dados de um usuário pelo email.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "O email do usuário a ser buscado."}
                },
                "required": ["email"]
            }
        }
    },
    "tool_003": {
        "tool_name": "Obter Volume de Vendas de um Produto",
        "description_for_llm": "Use esta ferramenta para consultar o volume total de vendas e o faturamento de um produto em um determinado período. Essencial para análises financeiras e de performance de vendas.",
        "api_spec": {
            "name": "get_sales_volume",
            "description": "Retorna o volume de vendas de um produto.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string", "description": "O nome do produto a ser consultado."},
                    "time_window": {"type": "string", "description": "A janela de tempo (ex: '7d', '30d', '90d')."}
                },
                "required": ["product_name", "time_window"]
            }
        }
    }
}

print(f"Catálogo criado com {len(tool_catalog)} ferramentas.\n")


--- 1. Definindo o Catálogo de Ferramentas (Simulação do DynamoDB) ---
Catálogo criado com 3 ferramentas.



In [ ]:
# ==============================================================================
# ETAPA 4-5: CONFIGURAR BANCO VETORIAL E CRIAR SCRIPT DE INDEXAÇÃO
# Usaremos FAISS em memória para simular o pgvector.
# O script de indexação gera os embeddings e os armazena no índice.
# ==============================================================================

print("--- 2. Indexando Ferramentas no Banco Vetorial (Simulação com FAISS) ---")

# bedrock_client = boto3.client("bedrock-runtime", region_name="us-east-1")
# embeddings_model = BedrockEmbeddings(client=bedrock_client, model_id="amazon.titan-embed-text-v2")

from langchain_core.embeddings import FakeEmbeddings

# Inicializa o FakeEmbeddings do LangChain (simulando dimensão 1024)
embeddings_model = FakeEmbeddings(size=1024)

# Extrai as descrições e os IDs das ferramentas para indexação
tool_ids = list(tool_catalog.keys())
tool_descriptions = [tool["description_for_llm"] for tool in tool_catalog.values()]

# Gera os embeddings para todas as descrições
print("Gerando embeddings fakes...")
vectors = embeddings_model.embed_documents(tool_descriptions)
print(f"Embeddings gerados. Dimensão do vetor: {len(vectors[0])}")

# Cria o índice FAISS e adiciona os vetores
try:
    import faiss
    dimension = len(vectors[0])
    # Usando IndexFlatL2 para busca de distância Euclidiana (padrão)
    vector_index = faiss.IndexFlatL2(dimension)
    vector_index.add(np.array(vectors, dtype=np.float32))
    print(f"Índice FAISS criado e populado com {vector_index.ntotal} vetores.\n")
except ImportError:
    print("FAISS não instalado. Pulando a criação do índice vetorial.")
    vector_index = None


--- 2. Indexando Ferramentas no Banco Vetorial (Simulação com FAISS) ---
Gerando embeddings fakes...
Embeddings gerados. Dimensão do vetor: 1024
Índice FAISS criado e populado com 3 vetores.



In [7]:
tool_descriptions

['Use esta ferramenta para obter a pontuação de Net Promoter Score (NPS) de um produto específico. A ferramenta pode receber um período de tempo (datas de início e fim) para filtrar a análise. É ideal para medir a lealdade e satisfação do cliente.',
 'Use esta ferramenta para recuperar informações detalhadas de um usuário, como nome, data de cadastro e status da conta, usando o email como identificador. É a ferramenta certa para quando a pergunta envolve um usuário específico.',
 'Use esta ferramenta para consultar o volume total de vendas e o faturamento de um produto em um determinado período. Essencial para análises financeiras e de performance de vendas.']

In [12]:
print(vectors[0])

[np.float64(-1.095799770048398), np.float64(-0.45537245892331774), np.float64(-1.4390775988301054), np.float64(0.04726368015765768), np.float64(0.46254327617758373), np.float64(0.30462692136219144), np.float64(2.1913227830317914), np.float64(-0.12327311575017792), np.float64(-2.473631755789654), np.float64(-0.793094686478074), np.float64(0.468258031926492), np.float64(-0.17021623615679798), np.float64(-1.0114386451273232), np.float64(-0.7192213391249244), np.float64(-1.062528361136806), np.float64(0.4939275048422632), np.float64(1.614594435853891), np.float64(-0.8647197645251669), np.float64(-0.09119449916249789), np.float64(-0.14576677520113754), np.float64(0.5038448164849184), np.float64(0.28584649787264693), np.float64(0.7840566773665237), np.float64(-0.4947421626417346), np.float64(-0.048050336913918704), np.float64(0.2874827928606266), np.float64(-1.5745939799069413), np.float64(-0.026829080234463586), np.float64(-1.0302096935743572), np.float64(-0.5557139233297497), np.float64(-0

In [13]:
# ==============================================================================
# ETAPA 6: CONSTRUIR O "SERVIÇO DE DESCOBERTA"
# Uma função que simula uma API que busca as ferramentas mais relevantes.
# ==============================================================================

print("--- 3. Construindo o Serviço de Descoberta de Ferramentas ---")

def discover_tools(user_query: str, top_k: int = 2):
    """
    Recebe a pergunta do usuário, gera seu embedding e busca as k ferramentas
    mais relevantes no nosso índice vetorial.
    """
    print(f"\n[Discovery Service] Recebida a query: '{user_query}'")
    
    # 1. Gera o embedding da pergunta do usuário
    query_vector = embeddings_model.embed_query(user_query)
    
    # 2. Busca no índice FAISS pelos vizinhos mais próximos
    distances, indices = vector_index.search(np.array([query_vector], dtype=np.float32), top_k)
    
    # 3. Mapeia os índices de volta para os tool_ids e retorna as ferramentas completas
    retrieved_tool_ids = [tool_ids[i] for i in indices[0]]
    retrieved_tools = [tool_catalog[tool_id] for tool_id in retrieved_tool_ids]
    
    print(f"[Discovery Service] Ferramentas mais relevantes encontradas: {[tool['tool_name'] for tool in retrieved_tools]}")
    return retrieved_tools

print("Serviço de Descoberta pronto.\n")

--- 3. Construindo o Serviço de Descoberta de Ferramentas ---
Serviço de Descoberta pronto.



In [15]:
# ==============================================================================
# ETAPA 7: TESTE DE RACIOCÍNIO - SIMULANDO O AGENTE
# Aqui, juntamos tudo: a query do usuário, o serviço de descoberta e o LLM.
# ==============================================================================

print("--- 4. Testando o Raciocínio do Agente com um LLM ---")

# llm = ChatBedrock(
#     client=bedrock_client,
#     model_id="anthropic.claude-3-sonnet-20240229-v1:0",
#     model_kwargs={"temperature": 0.0}
# )

from langchain_core.messages import AIMessage

# Criando uma classe Mock bem simples para simular o comportamento exato que esperamos do ChatBedrock.
# Isso evita os erros de validação do Pydantic que o FakeListChatModel possui com objetos AIMessage.
class FakeClaudeMock:
    def __init__(self, responses):
        self.responses = responses
        self.index = 0
        
    def bind_tools(self, tools):
        # Apenas retornamos a nós mesmos, ignorando o "bind"
        return self
        
    def invoke(self, query):
        response = self.responses[self.index]
        # Avança para a próxima resposta e volta ao início se acabar
        self.index = (self.index + 1) % len(self.responses)
        return response

dummy_responses = [
    # Teste 1: NPS
    AIMessage(content="", tool_calls=[{"name": "calculate_nps", "args": {"product_name": "SuperPlataforma"}, "id": "fake_call_1"}]),
    # Teste 2: Buscar Usuário
    AIMessage(content="", tool_calls=[{"name": "get_user_by_email", "args": {"email": "cliente@exemplo.com"}, "id": "fake_call_2"}]),
    # Teste 3: Volume de Vendas
    AIMessage(content="", tool_calls=[{"name": "get_sales_volume", "args": {"product_name": "Produto Alfa", "time_window": "30d"}, "id": "fake_call_3"}]),
    # Teste 4: Recusa (Nenhuma ferramenta acionada)
    AIMessage(content="Não consegui encontrar nenhuma ferramenta em meu catálogo para consultar a previsão do tempo amanhã, desculpe.")
]

llm = FakeClaudeMock(responses=dummy_responses)

def run_agent_reasoning(user_query: str):
    """
    Simula o fluxo completo do agente: descoberta -> raciocínio -> decisão.
    """
    print("="*50)
    print(f"EXECUTANDO AGENTE PARA A QUERY: '{user_query}'")
    print("="*50)

    # Passo 1: O agente usa o serviço de descoberta para encontrar ferramentas candidatas.
    candidate_tools = discover_tools(user_query)
    
    # Adiciona a capacidade de "tool calling" ao LLM com as ferramentas candidatas
    tools_for_llm = [tool['api_spec'] for tool in candidate_tools]
    
    llm_with_tools = llm.bind_tools(tools_for_llm)

    # Passo 2: O agente monta o prompt e pede ao LLM para escolher uma ferramenta.
    print("\n[Agente] Enviando pergunta e ferramentas candidatas para o LLM Mock...")
    ai_response = llm_with_tools.invoke(user_query)
    
    # Passo 3: O agente analisa a resposta do LLM e exibe a decisão.
    if not hasattr(ai_response, 'tool_calls') or not ai_response.tool_calls:
        print("\n[Agente] Decisão do LLM: Não foi necessário usar uma ferramenta. Resposta direta:")
        print(ai_response.content)
    else:
        print("\n[Agente] Decisão do LLM: Usar a seguinte ferramenta:")
        for tool_call in ai_response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"  - Nome da Ferramenta: {tool_name}")
            print(f"  - Argumentos: {json.dumps(tool_args, indent=2)}")


--- 4. Testando o Raciocínio do Agente com um LLM ---


In [16]:
# --- EXECUÇÃO DOS TESTES ---
# Agora, vamos testar o agente com diferentes perguntas.

# Teste 1: Pergunta claramente relacionada à ferramenta de NPS.
run_agent_reasoning("qual foi o nps do produto 'SuperPlataforma' no último mês?")

# Teste 2: Pergunta que deve acionar a busca de usuário.
run_agent_reasoning("Preciso saber o status da conta do usuário com email 'cliente@exemplo.com'.")

# Teste 3: Pergunta sobre vendas.
run_agent_reasoning("quanto vendemos do 'Produto Alfa' nos últimos 30 dias?")

# Teste 4: Pergunta que não corresponde a nenhuma ferramenta (teste de recusa).
run_agent_reasoning("qual a previsão do tempo para amanhã?")

EXECUTANDO AGENTE PARA A QUERY: 'qual foi o nps do produto 'SuperPlataforma' no último mês?'

[Discovery Service] Recebida a query: 'qual foi o nps do produto 'SuperPlataforma' no último mês?'
[Discovery Service] Ferramentas mais relevantes encontradas: ['Buscar Informações de Usuário', 'Calcular Net Promoter Score (NPS)']

[Agente] Enviando pergunta e ferramentas candidatas para o LLM Mock...

[Agente] Decisão do LLM: Usar a seguinte ferramenta:
  - Nome da Ferramenta: calculate_nps
  - Argumentos: {
  "product_name": "SuperPlataforma"
}
EXECUTANDO AGENTE PARA A QUERY: 'Preciso saber o status da conta do usuário com email 'cliente@exemplo.com'.'

[Discovery Service] Recebida a query: 'Preciso saber o status da conta do usuário com email 'cliente@exemplo.com'.'
[Discovery Service] Ferramentas mais relevantes encontradas: ['Calcular Net Promoter Score (NPS)', 'Obter Volume de Vendas de um Produto']

[Agente] Enviando pergunta e ferramentas candidatas para o LLM Mock...

[Agente] Decisão